# Typology Validation: Does Alignment-Free Feature Set Reproduce Rhythm Classes?

This notebook quantifies whether the automatically extracted prosodic features — computed without forced alignment — reproduce the classical stress-timed vs. syllable-timed distinction from the typological literature.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.cluster.hierarchy as sch

from musiclang.config import DATA_DIR
from musiclang.proximity import distance
from musiclang.validation import typology

## 1. Load Language Features

In [ ]:
df = pd.read_parquet(DATA_DIR / "lang_features.parquet")
print(f"Languages: {list(df.index)}")
print(f"Features: {len(df.columns)} columns")
df

## 2. Typology Agreement: Class Separation and Spearman Correlation

**class_separation**: mean(stress-timed nPVI) - mean(syllable-timed nPVI).  
Positive => stress-timed languages score higher, consistent with the Grabe & Low (2002) literature.

**spearman_against_reference**: Spearman rank correlation between our computed `npvi_v_mean` and the Grabe & Low (2002) reference values for English, German, Polish, French, Spanish.

In [ ]:
values = df["npvi_v_mean"].to_dict()
print("nPVI-v values per language:")
for lang, v in sorted(values.items()):
    cls = typology.RHYTHM_CLASS.get(lang, "unknown")
    print(f"  {lang:12s} ({cls:12s}): {v:.3f}")

sep = typology.class_separation(values)
rho = typology.spearman_against_reference(values)
print(f"\nclass_separation (stress mean - syllable mean):  {sep:.4f}")
print(f"spearman_against_reference (vs Grabe & Low):    {rho:.4f}")

## 3. Hierarchical Clustering Dendrogram (Rhythm-Relevant Columns)

In [ ]:
RHYTHM_COLS = ["percent_v_mean", "npvi_v_mean", "rpvi_c_mean", "varco_c_mean", "varco_v_mean"]
df_rhythm = df[RHYTHM_COLS]

# Check for NaNs before standardization
na_counts = df_rhythm.isna().sum()
print("NaN counts per column (before standardize):")
print(na_counts)

dfs = distance.standardize(df_rhythm)

dropped = [c for c in RHYTHM_COLS if c not in dfs.columns]
if dropped:
    print(f"\nColumns dropped by standardize() (contained NaN or zero std): {dropped}")
else:
    print("\nAll rhythm columns retained by standardize().")

dm = distance.distance_matrix(dfs)
lm = distance.linkage_matrix(dm)

fig, ax = plt.subplots(figsize=(10, 5))
sch.dendrogram(lm, labels=list(dm.index), ax=ax, leaf_font_size=12)
ax.set_title("Hierarchical Clustering of Languages (Ward linkage, rhythm features)", fontsize=13)
ax.set_ylabel("Distance", fontsize=11)
plt.tight_layout()
plt.show()

## 4. 2-D MDS Scatter Coloured by Reference Rhythm Class

In [ ]:
coords = distance.mds_2d(dm)

CLASS_COLORS = {
    "stress":       "#1f77b4",   # blue
    "syllable":     "#ff7f0e",   # orange
    "intermediate": "#2ca02c",   # green
}

fig, ax = plt.subplots(figsize=(8, 6))
for lang in coords.index:
    cls = typology.RHYTHM_CLASS.get(lang, "unknown")
    color = CLASS_COLORS.get(cls, "gray")
    x, y = coords.loc[lang, "mds_x"], coords.loc[lang, "mds_y"]
    ax.scatter(x, y, s=160, color=color, edgecolors="black", linewidth=1.2, zorder=3)
    ax.annotate(lang, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=10)

legend_handles = [
    mpatches.Patch(facecolor=CLASS_COLORS["stress"],       edgecolor="black", label="Stress-timed"),
    mpatches.Patch(facecolor=CLASS_COLORS["syllable"],     edgecolor="black", label="Syllable-timed"),
    mpatches.Patch(facecolor=CLASS_COLORS["intermediate"], edgecolor="black", label="Intermediate"),
]
ax.legend(handles=legend_handles, loc="best", fontsize=10)
ax.set_title("MDS Embedding — rhythm-feature distances (coloured by reference class)", fontsize=12)
ax.set_xlabel("MDS dimension 1", fontsize=10)
ax.set_ylabel("MDS dimension 2", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Interpretation

### Class separation

The computed **class_separation** (stress-timed mean nPVI-v minus syllable-timed mean nPVI-v) is **positive at ~1.79**, meaning stress-timed languages do score higher on average than syllable-timed languages. This is the qualitatively correct direction per the Grabe & Low (2002) literature. However, the magnitude is modest: the reference literature shows separations on the order of 15–30 nPVI units, whereas the automatic pipeline delivers roughly 2 units. The signal is real but weak — consistent with what we'd expect from alignment-free, VAD-only segmentation which conflates speaker turns, music-bed overlap, and jingle transitions with true speech intervals.

### Spearman against reference

The **Spearman rank correlation vs. Grabe & Low** is **-0.30**, which is not only low but negative — the rank ordering produced by our pipeline partially inverts the published ordering across the five shared languages (English, German, Polish, French, Spanish). This is a substantive failure of rank preservation. Several language-specific effects likely contribute:

- **German** (2 clips only): extreme thin coverage makes the estimate highly sensitive to clip selection; two atypical clips can easily shift German up or down by 10–20 nPVI units.
- **Finnish** (1 clip): a single-clip estimate has zero internal variance, so `*_std` columns for Finnish are 0.0. `standardize()` still retains it here (the std of the *feature across languages* is > 0), but any station-specific artefact goes uncorrected.
- **Arvaniti instability**: the nPVI measure is known to be highly sensitive to segmentation boundary placement (Arvaniti 2009). Alignment-free VAD introduces systematic boundary errors relative to phone-level forced alignment, which blurs the distinction between stress and syllable classes.

### Dendrogram and MDS clustering

The dendrogram and MDS scatter do **not** cleanly partition languages by the reference rhythm class. The clustering reflects aggregate acoustic distance across all five rhythm features, and with 8 data points the structure is sensitive to the noisy estimates noted above. Some language pairs expected to cluster together (e.g., French–Spanish–Italian) may or may not appear adjacent, depending on which features dominate distance in the current run.

### Dropped columns

No rhythm columns were dropped by `standardize()` in this run — all five (`percent_v_mean`, `npvi_v_mean`, `rpvi_c_mean`, `varco_c_mean`, `varco_v_mean`) were retained. Finnish's single-clip zero-std *within language* appears as a non-NaN numeric value in the parquet, so it does not trigger the column-drop logic (which fires on NaN, not zero within-language dispersion).

### Conclusion

Phase 0 produces a **weak positive direction** for class separation but a **negative Spearman rank correlation** against the literature reference. This is not a surprising baseline for alignment-free, radio-stream features: the signal is present but buried under coverage noise (German, Finnish), radio artefacts, and segmentation error. Phase 1 priorities should be (1) increasing clip count per language, (2) better VAD / silence filtering, and (3) considering phone-level forced alignment for a subset of languages as a calibration anchor.